# Some WiP feats

We are also working on some more feats:
- Adding support for multi-target ([#2356](https://github.com/sktime/pytorch-forecasting/pull/2356) and [#2324](https://github.com/sktime/pytorch-forecasting/pull/2324))
- Improving the serialisation interface for v2 ([#2323](https://github.com/sktime/pytorch-forecasting/pull/2323))
- Hyperparam optimization ([#2335](https://github.com/sktime/pytorch-forecasting/pull/2335))
- Categorical support ([#2082](https://github.com/sktime/pytorch-forecasting/pull/2082))
- Adding temporal splitting ([#2087](https://github.com/sktime/pytorch-forecasting/pull/2087))
- Adding support of `nn` losses ([#2331](https://github.com/sktime/pytorch-forecasting/pull/2331))

As most of the above mentioned feats are still WiP and the design is yet to be finalised, we shall look at just one feature, that is very close to completion and can be benifitted from your input!

Lets have a look at how `save` and `load` (will/may) work in v2

Install the branch first :)

In [1]:
!pip install git+https://github.com/phoeenniixx/pytorch-forecasting.git@save-and-load

  Cloning https://github.com/phoeenniixx/pytorch-forecasting.git (to revision save-and-load) to /tmp/pip-req-build-zevhpbt5
  Running command git clone --filter=blob:none --quiet https://github.com/phoeenniixx/pytorch-forecasting.git /tmp/pip-req-build-zevhpbt5
  Running command git checkout -b save-and-load --track origin/save-and-load
  Switched to a new branch 'save-and-load'
  Branch 'save-and-load' set up to track remote branch 'save-and-load' from 'origin'.
  Resolved https://github.com/phoeenniixx/pytorch-forecasting.git to commit 22e9d77632b8346fcce1cf936e85d16cf52fcc8b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### Lets train a model first

1. Create the same toy dataset

In [2]:
# Copied here from utils to prevent any issues in colab!
import numpy as np
import pandas as pd
def load_toydata(num_series, seq_length):
    data_list = []
    for i in range(num_series):
        x = np.arange(seq_length)
        level = 10 ** (i % 3)
        y = level * np.sin(x / 5.0) + np.random.normal(scale=0.1, size=seq_length)
        category = i % 5
        static_value = np.random.rand()
        for t in range(seq_length - 1):
            data_list.append(
                {
                    "series_id": i,
                    "time_idx": t,
                    "x": y[t],
                    "y": y[t + 1],
                    "category": category,
                    "future_known_feature": np.cos(t / 10),
                    "static_feature": static_value,
                    "static_feature_cat": i % 3,
                }
            )
    data_df = pd.DataFrame(data_list)
    return data_df

In [3]:
from pytorch_forecasting.data.data_module import EncoderDecoderTimeSeriesDataModule
from pytorch_forecasting.data.timeseries import TimeSeries

In [4]:
# Play around with the toy dataset
num_series = 100
seq_length = 50
df = load_toydata(num_series, seq_length)
df.head()

,series_id,time_idx,x,y,category,future_known_feature,static_feature,static_feature_cat
0,0,0,-0.216377,0.243427,0,1.000000,0.532392,0
1,0,1,0.243427,0.263074,0,0.995004,0.532392,0
2,0,2,0.263074,0.529324,0,0.980067,0.532392,0
3,0,3,0.529324,0.695787,0,0.955336,0.532392,0
4,0,4,0.695787,0.933469,0,0.921061,0.532392,0


2. Create `TimeSeries` class

In [5]:
# create `TimeSeries` dataset that returns the raw data in terms of tensors
dataset = TimeSeries(
    data=df,
    time="time_idx",
    target="y",
    group=["series_id"],
    num=["x", "future_known_feature", "static_feature"],
    cat=["category", "static_feature_cat"],
    known=["future_known_feature"],
    unknown=["x", "category"],
    static=["static_feature", "static_feature_cat"],
)

/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/data/timeseries/_timeseries_v2.py:104: UserWarning: TimeSeries is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(


3. Create cfgs and pkg class

In [6]:
from sklearn.preprocessing import StandardScaler
from pytorch_forecasting.data.encoders import TorchNormalizer
datamodule_cfg = dict(
    max_encoder_length=30,
    max_prediction_length=1,
    batch_size=32,
    scalers={
        "x": StandardScaler(),
        "future_known_feature": StandardScaler(),
        "static_feature": StandardScaler(),
    },
    target_normalizer=TorchNormalizer(),
)

In [7]:
from pytorch_forecasting.models.samformer import Samformer_pkg_v2
from pytorch_forecasting.metrics import QuantileLoss

In [8]:
model_cfg=dict(
    hidden_size=512,
    loss=QuantileLoss(quantiles=[0.1, 0.5, 0.9]),
    use_revin=True,
    persistence_weight=0.1
)

In [9]:
trainer_cfg = dict(
    max_epochs=5,
    accelerator="auto",
    devices=1,
    enable_progress_bar=True,
    log_every_n_steps=10,
)

In [10]:
model_pkg = Samformer_pkg_v2(
    model_cfg=model_cfg,
    trainer_cfg=trainer_cfg,
    datamodule_cfg=datamodule_cfg,
)

In [11]:
!mkdir checkpoints

During `fit`, we also pass `ckpt_dir` where we want to save the checkpoints (and other artifacts)

In [12]:
model_pkg.fit(dataset, ckpt_dir="/content/checkpoints")

/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/data/data_module/_encoder_decoder_data_module.py:158: UserWarning: EncoderDecoderTimeSeriesDataModule is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(
/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/models/base/_base_model_v2.py:85: UserWarning: The Model 'Samformer' is part of an experimental reworkof the pytorch-forecasting model layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. This class is intended for beta testing and as a basic skeleton, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, 

┏━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name              ┃ Type         ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss              │ QuantileLoss │      0 │ train │     0 │
│ 1 │ logging_metrics   │ ModuleList   │      0 │ train │     0 │
│ 2 │ revin             │ RevIN        │      8 │ train │     0 │
│ 3 │ compute_keys      │ Linear       │ 15.9 K │ train │     0 │
│ 4 │ compute_queries   │ Linear       │ 15.9 K │ train │     0 │
│ 5 │ compute_values    │ Linear       │    930 │ train │     0 │
│ 6 │ linear_forecaster │ Linear       │     31 │ train │     0 │
└───┴───────────────────┴──────────────┴────────┴───────┴───────┘

Trainable params: 32.7 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 32.7 K                                                                                               
Total estimated model params size (MB): 0.131                                                                      
Modules in train mode: 7                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO: `Trainer.fit` stopped: `max_epochs=5` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.


Artifacts saved in: /content/checkpoints


PosixPath('/content/checkpoints/checkpoints/best-epoch=4-step=210.ckpt')

Lets see what the checkpoints directory has exactly?

In [13]:
!ls checkpoints

artifacts.yaml	checkpoints  configs  metadata	scalers


you can look all the artifacts that have been saved in the folder, by looking at the `artifacts.yaml` file

In [14]:
import yaml
from rich.pretty import pprint

with open('checkpoints/artifacts.yaml', 'r') as file:
    yaml_content = yaml.safe_load(file)

pprint(yaml_content)

{
│   'artifacts': {
│   │   'best_model_checkpoint': '/content/checkpoints/checkpoints/best-epoch=4-step=210.ckpt',
│   │   'datamodule_cfg': '/content/checkpoints/configs/datamodule_cfg.pkl',
│   │   'datamodule_metadata': '/content/checkpoints/metadata/datamodule_metadata.pkl',
│   │   'last_model_checkpoint': '/content/checkpoints/checkpoints/last.ckpt',
│   │   'model_cfg': '/content/checkpoints/configs/model_cfg.pkl',
│   │   'scaler': '/content/checkpoints/scalers/scaler.pkl',
│   │   'target_normalizer': '/content/checkpoints/scalers/target_normalizer.pkl',
│   │   'trainer_cfg': '/content/checkpoints/configs/trainer_cfg.pkl'
│   }
}

Lets load this model and perform the predictions

In [15]:
loaded_model_pkg=Samformer_pkg_v2.load("/content/checkpoints")

INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [16]:
preds=loaded_model_pkg.predict(dataset)
pprint(preds)

/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/data/data_module/_encoder_decoder_data_module.py:158: UserWarning: EncoderDecoderTimeSeriesDataModule is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(
INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments 

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


{
│   'prediction': tensor([[-0.1757],
│   │   [-0.1041],
│   │   [-0.0448],
│   │   ...,
│   │   [-0.1625],
│   │   [-0.2265],
│   │   [-0.1556]])
}

You can also choose what to "exclude" while saving by passing `exclude` to `fit`

In [17]:
!mkdir exclude_checkpoints

In [18]:
model_pkg.fit(dataset, ckpt_dir="/content/exclude_checkpoints", exclude=["scaler"])

/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/data/data_module/_encoder_decoder_data_module.py:158: UserWarning: EncoderDecoderTimeSeriesDataModule is part of an experimental rework of the pytorch-forecasting data layer, scheduled for release with v2.0.0. The API is not stable and may change without prior warning. For beta testing, but not for stable production use. Feedback and suggestions are very welcome in pytorch-forecasting issue 1736, https://github.com/sktime/pytorch-forecasting/issues/1736
  warn(
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /content/checkpoints/checkpoints exists and is not empty.


┏━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name              ┃ Type         ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss              │ QuantileLoss │      0 │ train │     0 │
│ 1 │ logging_metrics   │ ModuleList   │      0 │ train │     0 │
│ 2 │ revin             │ RevIN        │      8 │ train │     0 │
│ 3 │ compute_keys      │ Linear       │ 15.9 K │ train │     0 │
│ 4 │ compute_queries   │ Linear       │ 15.9 K │ train │     0 │
│ 5 │ compute_values    │ Linear       │    930 │ train │     0 │
│ 6 │ linear_forecaster │ Linear       │     31 │ train │     0 │
└───┴───────────────────┴──────────────┴────────┴───────┴───────┘

Trainable params: 32.7 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 32.7 K                                                                                               
Total estimated model params size (MB): 0.131                                                                      
Modules in train mode: 7                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO: `Trainer.fit` stopped: `max_epochs=5` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.


/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/callbacks/artifact_registry.py:127: UserWarning: Key 'best_model_checkpoint' not found in /content/exclude_checkpoints/artifacts.yaml.
  warnings.warn(f"Key '{key}' not found in {registry_path}.")


Now scalers are not present!

In [20]:
import yaml
from rich.pretty import pprint

with open('exclude_checkpoints/artifacts.yaml', 'r') as file:
    yaml_content = yaml.safe_load(file)

pprint(yaml_content)

{
│   'artifacts': {
│   │   'datamodule_cfg': '/content/exclude_checkpoints/configs/datamodule_cfg.pkl',
│   │   'datamodule_metadata': '/content/exclude_checkpoints/metadata/datamodule_metadata.pkl',
│   │   'model_cfg': '/content/exclude_checkpoints/configs/model_cfg.pkl',
│   │   'target_normalizer': '/content/exclude_checkpoints/scalers/target_normalizer.pkl',
│   │   'trainer_cfg': '/content/exclude_checkpoints/configs/trainer_cfg.pkl'
│   }
}

You can also decide to `skip` loading something, by pasing `skip` to `load`

In [22]:
skipped_model_pkg = Samformer_pkg_v2.load("/content/checkpoints", skip=["scaler"])

INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/models/base/_base_model_v2.py:85: UserWarning: The Model 'Samformer' is part of an experimental reworkof the pytorch-forecasting model layer, scheduled for release with v2.0.0. The AP

In [23]:
skipped_model_pkg.datamodule._scalers

AttributeError: 'NoneType' object has no attribute '_scalers'

In [24]:
model_pkg.datamodule._scalers

{'x': <pytorch_forecasting.adapters.scaler_adapters.ScalerAdapter at 0x7e60ecb679e0>,
 'future_known_feature': <pytorch_forecasting.adapters.scaler_adapters.ScalerAdapter at 0x7e6080432a80>,
 'static_feature': <pytorch_forecasting.adapters.scaler_adapters.ScalerAdapter at 0x7e6080359790>}

### Open Question
 
- Should we have `include` in `save` (along with `exclude`) which allows you to "include" the artifacts. This will give the users freedom to choose the list which is shorter - they can "include" the things if they want 2-3 artifacts only or can "exclude" the things if they want to exclude 2-3 things?
- Similar idea in `load`?